In [1]:
print("Jay Ganesh")

Jay Ganesh


## Hybrid Search Langchain

In [2]:
import os
from dotenv import load_dotenv
load_dotenv()

True

In [3]:
pinecone_api=os.getenv("PINECONE_API_KEY")

In [4]:
from langchain_community.retrievers import PineconeHybridSearchRetriever

In [5]:
from pinecone import Pinecone,ServerlessSpec

In [7]:
index_name="hybrid-search-langchain-pinecone"
#initialize the pinecone client
pc=Pinecone(api_key=pinecone_api)


if index_name not in pc.list_indexes().names():
    pc.create_index(name=index_name,dimension=384,metric='dotproduct',spec=ServerlessSpec(cloud='aws',region='us-east-1'))

In [9]:
index=pc.Index(index_name)
index

In [10]:
from langchain_huggingface import HuggingFaceEmbeddings
embeddings=HuggingFaceEmbeddings(model="all-MiniLM-L6-v2")
embeddings

HuggingFaceEmbeddings(model_name='all-MiniLM-L6-v2', cache_folder=None, model_kwargs={}, encode_kwargs={}, query_encode_kwargs={}, multi_process=False, show_progress=False)

In [12]:
from pinecone_text.sparse import BM25Encoder
bm25=BM25Encoder().default()
bm25

In [13]:
sentences=[
    "In 2023, I visited Goa",
    "In 2024, I visited Kerala",
    "In 2025, I vistied who cares"
]

In [14]:
bm25.fit(sentences)

100%|████████████████████████████████████████████████████████████████████████████████████| 3/3 [00:00<00:00, 24.91it/s]


In [16]:
bm25.dump("bm25_vales.json")

In [19]:
bm25=BM25Encoder().load("bm25_vales.json")

In [21]:
retriever=PineconeHybridSearchRetriever(embeddings=embeddings ,sparse_encoder=bm25,index=index)

In [22]:
retriever.add_texts(
    [
        "In 2023, I visited Goa",
    "In 2024, I visited Kerala",
    "In 2025, I vistied who cares"
    ]
)

100%|████████████████████████████████████████████████████████████████████████████████████| 1/1 [00:02<00:00,  2.81s/it]


In [26]:
retriever.invoke("who cares")

[Document(metadata={'score': 0.486676931}, page_content='In 2025, I vistied who cares'),
 Document(metadata={'score': 0.0760703087}, page_content='In 2024, I visited Kerala'),
 Document(metadata={'score': 0.0555005074}, page_content='In 2023, I visited Goa')]